# Geração de Sequência de Proteínas com Modelo de Transformadores

In [2]:
# Imports
import time
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [3]:
# Define o device
device = torch.device("cpu" if torch.backends.mps.is_built() else "cuda")
print("Device:", device)

Device: cuda


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


/home/priscila/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 9010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Dados

Usaremos dados de sequências de proteínas, onde cada proteína é descrita por uma sequência de  Aminoácidos (AA).
Cada  aminoácido  exclusivo  é  representado  por  uma  letra exclusiva do alfabeto. 

Veja aqui o que cada letra representa: 

https://www.bioinformatics.org/sms/iupac.html

Exemplo do formato dos dados: MAFLKKSLFLVLFLGVVSLSFCEEEKREEHEEEKRDEEDAESLGKR

As  proteínas  são  de  eucariontes,  filtradas  para  ter comprimento de 40-200 AA, e foram originalmente retiradas do site UniProt: https://www.uniprot.org

### Carrega dados

In [5]:
def carrega_dados_json(filename):
    
    with open(filename) as f:
        
        data = json.load(f)

    return data

In [6]:
sequences = carrega_dados_json("dados.json")

In [7]:
print(f"Número de sequências de proteínas: {len(sequences)}")

Número de sequências de proteínas: 42980


In [8]:
sequences[30:34]

['MRTLWIMAVLLLGVEGDLMQFETLIMKIAKRSGMFWYSAYGCYCGWGGQGRPQDATDRCCFVHDCCYGKVTGCDPKLDSYTYSVENGDVVCGGNDPCKKEICECDRAAAICFRDNKVTYDNKYWRFPPQNCKEESEPC',
 'MKLLAPVLWAMAALGVTWLVAVDSKESCTKHSNGCSTPLRLPCQEYFRPACDIHDNCYHCGTIFGISRKECDDAFLKDMNTLCKKLGSNSATCPARGKREVTSHRATSIAHSRLWKTALDQKSFLNRKARQAILLTPNSCLYWANNFYMAVHVFGARSYSRTTDPKDCQGLKHCLPNH',
 'MKFIIVLLLLTALTLTSIPVIEGILKRCKTYDDCKDVCKARKGKCEFGICKCMIKSGK',
 'MLLYICLVNLLLPLSVGAASGAALGVIAKVGVDAALQQIDDVWKGKTVRYWKCAVENRSSKTLYALGTTQESGSMTTVFADIPPKSTGVFVWEKSRGAAKGAVGVVHYKYGNKVLNIMASIPYDWNLYKAWANVHLSDHKESFSDLYKGKNGAKYPTRAGNWGEVDGTKFFLTEKSHAEFKVIFSG']

### Exploração dos dados

In [9]:
# Verifica o tamanho (len) de cada sequência
seq_lens = [len(seq) for seq in sequences]

In [10]:
print(f"Comprimento Mínimo: {min(seq_lens)}")
print(f"Comprimento Médio: {sum(seq_lens) / len(seq_lens)}")
print(f"Comprimento Máximo: {max(seq_lens)}")

Comprimento Mínimo: 40
Comprimento Médio: 125.96361098185203
Comprimento Máximo: 200


In [11]:
# Função que recebe uma lista de sequências como argumento
def get_token_freq(sequences):
    
    # Inicializa um dicionário vazio para armazenar a frequência dos tokens
    token_freq = {}
    
    # Itera sobre cada sequência
    for seq in sequences:
        
        # Itera sobre cada token na sequência atual
        for token in seq:
            
            if token not in token_freq:
                
                # Adiciona o token ao dicionário com uma contagem inicial de 0
                token_freq[token] = 0
            
            # Incrementa a contagem de frequência do token em 1
            token_freq[token] += 1
    
    # Retorna o dicionário contendo as frequências dos tokens
    return token_freq

In [12]:
aa_freq = get_token_freq(sequences);aa_freq

{'M': 144705,
 'A': 395518,
 'F': 224599,
 'L': 506986,
 'K': 374175,
 'S': 397801,
 'V': 342011,
 'G': 365247,
 'C': 159924,
 'E': 330494,
 'R': 305259,
 'H': 117541,
 'D': 254761,
 'Y': 167821,
 'P': 260364,
 'I': 283222,
 'T': 286033,
 'N': 222329,
 'Q': 208619,
 'W': 64812,
 'U': 73,
 'X': 1183,
 'Z': 220,
 'B': 219}

Ordena as chaves do dicionário freq com base nos valores associados a essas chaves, do maior para o menor

In [13]:
aa_sorted = sorted(aa_freq.keys(), key = lambda x: aa_freq[x], reverse = True)  

In [14]:
print(f"Número de aminoácidos diferentes: {len(aa_freq)}")
print(f"Número total de aminoácidos: {sum(aa_freq.values()):,}")

Número de aminoácidos diferentes: 24
Número total de aminoácidos: 5,413,916


In [ ]:
# Plot
plt.figure(figsize = (12, 5))
plt.title("Frequência de Aminoácidos")
plt.bar(range(len(aa_freq)), 
        [aa_freq[aa] for aa in aa_sorted], 
        align = "center", 
        tick_label = aa_sorted, 
        color = 'pink')
plt.xlabel("Aminoácidos")
plt.ylabel("Frequência")
plt.show()

In [16]:
print(aa_sorted)

['L', 'S', 'A', 'K', 'G', 'V', 'E', 'R', 'T', 'I', 'P', 'D', 'F', 'N', 'Q', 'Y', 'C', 'M', 'H', 'W', 'X', 'Z', 'B', 'U']


### Preparação dos dados - Construindo o vocabulário

In [17]:
# Cria o vocabulário
vocab = ["<pad>", "<bos>", "<eos>"] + aa_sorted

In [18]:
vocab_size = len(vocab)

print(f"Tamanho do Vocabulário: {vocab_size}")

Tamanho do Vocabulário: 27


In [19]:
print(f"Vocabulário:", *vocab)

Vocabulário: <pad> <bos> <eos> L S A K G V E R T I P D F N Q Y C M H W X Z B U


In [20]:
# Construir mapeamento de token para índices (e vice-versa)
token2idx = {token: i for i, token in enumerate(vocab)}
idx2token = {i: token for i, token in enumerate(vocab)}

### Preparação dos Dados - Classe para estrutura de dados

#### Padding

O **padding** no refere-se ao processo de adicionar tokens especiais (neste caso, o token
`<pad>`) ao final das sequências de entrada (input_ids) e de saída (target_ids) de modo a torná-las todas de um mesmo comprimento.

Em redes neurais que processam sequências de dados, como aquelas usadas em PLN, é comum que as sequências (como frases ou sentenças) tenham tamanhos variados. No entanto, os modelos de Deep Learning geralmente requerem que todas as sequências dentro de um batch tenham o mesmo comprimento, pois trabalham com matrizes de tamanho fixo.

O padding resolve esse problema ao:

- Encontrar a sequência mais longa do batch (max_len).
- Adicionar o token de padding (`<pad>`) no final das sequências menores para que todas tenham o mesmo comprimento.

No caso específico abaixo, o padding é aplicado tanto aos input_ids (que são os IDs dos tokens de entrada) quanto aos target_ids (os IDs dos tokens de saída), e o token de padding é representado por 
`self.token2idx["<pad>"]`, que é o índice correspondente ao token de padding no vocabulário.

Isso garante que todas as sequências dentro de um batch tenham o mesmo tamanho, permitindo que sejam processadas de forma eficiente por redes neurais.

In [21]:
# Classe ProteinDataset que herda de Dataset
class ProteinDataset(Dataset):
    
    # Inicializa a classe com uma lista de sequências e um dicionário token2idx
    def __init__(self, sequences, token2idx):
        
        # Armazena o mapeamento token para índice
        self.token2idx = token2idx
        
        # Inicializa listas vazias para armazenar os dados de sequência e os IDs dos dados
        self.data, self.data_id = [], []
        
        # Itera sobre cada sequência na lista de sequências
        for seq in sequences:
            
            # Adiciona tokens de início (<bos>) e fim (<eos>) à sequência
            tokens = ["<bos>"] + [token for token in seq] + ["<eos>"]
            
            # Adiciona a sequência de tokens à lista de dados
            self.data.append(tokens)
            
            # Converte os tokens da sequência para IDs usando o mapeamento e adiciona à lista de IDs
            self.data_id.append([token2idx[token] for token in self.data[-1]])

    # Retorna o tamanho do conjunto de dados
    def __len__(self):
        return len(self.data)

    # Retorna um item específico do conjunto de dados com base no índice
    def __getitem__(self, index):
        
        # Define input_ids como todos os IDs, exceto o último
        input_ids = self.data_id[index][:-1]
        
        # Define target_ids como todos os IDs, exceto o primeiro
        target_ids = self.data_id[index][1:]

        # Retorna input_ids e target_ids como uma tupla
        return input_ids, target_ids

    # Função para aplicar padding em um batch
    def padding_batch(self, batch):
        
        # Separa os input_ids e target_ids para cada item no batch
        input_ids = [d[0] for d in batch]
        target_ids = [d[1] for d in batch]

        # Calcula o comprimento máximo entre as sequências input_ids
        max_len = max([len(i) for i in input_ids])

        # Itera sobre cada sequência para aplicar o padding
        for i in range(len(input_ids)):
            
            # Adiciona tokens <pad> ao final de cada sequência input_ids para igualar o comprimento
            input_ids[i] += [self.token2idx["<pad>"]] * (max_len - len(input_ids[i]))
            
            # Adiciona tokens <pad> ao final de cada sequência target_ids para igualar o comprimento
            target_ids[i] += [self.token2idx["<pad>"]] * (max_len - len(target_ids[i]))

        # Converte input_ids para um tensor do tipo LongTensor
        input_ids = torch.LongTensor(input_ids)
        
        # Converte target_ids para um tensor do tipo LongTensor
        target_ids = torch.LongTensor(target_ids)

        # Retorna os tensores input_ids e target_ids
        return input_ids, target_ids

`input_ids = self.data_id[index][:-1]:`

Aqui, `input_ids` representa a sequência de entrada para o modelo, incluindo todos os tokens da sequência, exceto o último. O modelo utiliza essa sequência para prever o próximo token em cada posição.

`target_ids = self.data_id[index][1:]:`

Em `target_ids`, são incluídos todos os tokens, exceto o primeiro. Essa lista representa a sequência de saída ou a sequência-alvo para o modelo, com cada token alinhado como a "próxima palavra" para os tokens correspondentes em input_ids.

O objetivo dessa configuração é permitir que o modelo aprenda a prever o próximo token em uma sequência. Por exemplo:

Para uma sequência `["<bos>", "A", "B", "C", "<eos>"]`:

`input_ids: ["<bos>", "A", "B", "C"]`

`target_ids: ["A", "B", "C", "<eos>"]`
    
Isso permite que o modelo receba `<bos>` e aprenda a prever A, depois receba A para prever B, e assim por diante, até o último token (C) prever `<eos>`. 

In [22]:
# Testando a classe
dataset = ProteinDataset(sequences[:10], token2idx)

In [23]:
# Testando o dataset
idx = 0
input_ids, target_ids = dataset[idx]
input_tokens, target_tokens = [idx2token[i] for i in input_ids], [idx2token[i] for i in target_ids]

In [24]:
print(f"Sequência original:", *sequences[idx])
print(f"Sequência de entrada:", *input_tokens)
print(f"Sequência de destino:", *target_tokens)

Sequência original: M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D
Sequência de entrada: <bos> M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D
Sequência de destino: M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D <eos>


#### Testando a função de preenchimento de lote com um lote fictício

In [25]:
batch = [
    [token2idx[i] for i in ['<bos>', 'A', 'B', 'C', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', 'B', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', 'B', 'C', 'D', '<eos>']],
]

batch

[[1, 5, 25, 19, 2], [1, 5, 25, 2], [1, 5, 2], [1, 5, 25, 19, 14, 2]]

In [26]:
# Gera o batch
batch2 = [(b[:-1], b[1:]) for b in batch]
batch2

[([1, 5, 25, 19], [5, 25, 19, 2]),
 ([1, 5, 25], [5, 25, 2]),
 ([1, 5], [5, 2]),
 ([1, 5, 25, 19, 14], [5, 25, 19, 14, 2])]

In [27]:
# Gera os ids
input_ids, target_ids = dataset.padding_batch(batch2)

In [28]:
print("Input batch:")
print(*[[idx2token[i] for i in x] for x in input_ids.tolist()], sep="\n")
print("\nTarget batch:")
print(*[[idx2token[i] for i in x] for x in target_ids.tolist()], sep="\n")

Input batch:
['<bos>', 'A', 'B', 'C', '<pad>']
['<bos>', 'A', 'B', '<pad>', '<pad>']
['<bos>', 'A', '<pad>', '<pad>', '<pad>']
['<bos>', 'A', 'B', 'C', 'D']

Target batch:
['A', 'B', 'C', '<eos>', '<pad>']
['A', 'B', '<eos>', '<pad>', '<pad>']
['A', '<eos>', '<pad>', '<pad>', '<pad>']
['A', 'B', 'C', 'D', '<eos>']


### Preparação dos dados - Configuração da Máscara

O token de padding (`<pad>`) não contém informações úteis e não deve ser considerado pelo modelo ao calcular as atenções (o mecanismo central do Transformer). Para isso, cria-se a **máscara de padding**, que define quais tokens são reais (não `<pad>`) e quais são padding, ignorando assim os tokens de padding durante a atenção. 

No caso de tarefas de geração de texto (como tradução automática ou modelagem de linguagem), o Transformer precisa garantir que, ao prever o próximo token em uma sequência, ele só possa "ver" os tokens anteriores, não os futuros. Isso é fundamental, pois o modelo não deve ter acesso a tokens futuros durante o treinamento.

Para implementar essa restrição, usamos a máscara causal, que é uma máscara triangular inferior, garantindo que cada token só possa "atentar" (gerar atenção) a si mesmo e aos tokens que o precedem na sequência. 

In [29]:
def get_mask(input_ids, pad_idx):
    
    seq_len = input_ids.shape[-1]                            # Obtém o comprimento da sequência a partir da última dimensão de input_ids

    mask_pad = input_ids != pad_idx                          # Cria uma máscara que é True para todos os tokens que não são <pad>
        
    mask_pad = mask_pad.unsqueeze(1).expand(-1, seq_len, -1) # Expande a máscara para ter uma dimensão extra, permitindo cobrir toda a sequência

                                                             # Cria uma máscara causal triangular inferior para garantir que cada token só "veja" tokens anteriores
    mask_causal = torch.tril(torch.ones(seq_len, seq_len, device = input_ids.device)).bool()

    mask = mask_pad & mask_causal                            # Combina a máscara de padding e a máscara causal para aplicar ambas restrições simultaneamente

    return mask.to(int)                                      # Converte a máscara para inteiros (0 e 1) e a retorna

In [30]:
# cria o tensor diretamente no device selecionado
input_ids = torch.ones(1, 5, dtype=torch.long, device=device)

In [31]:
input_ids

tensor([[1, 1, 1, 1, 1]])

In [32]:
# Cria a máscara
mask = get_mask(input_ids, token2idx["<pad>"])
mask

tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]])

## Divisão dos dados em Treino e Teste

In [33]:
# Define uma função que divide as sequências em conjuntos de treino, validação e teste
def split_sequences(sequences, train_ratio, valid_ratio, subsample = None, seed = 42):

    # Define a semente para o gerador de números aleatórios do NumPy para garantir replicabilidade
    np.random.seed(seed)
    
    # Cria um array de índices do tamanho do número de sequências
    indices = np.arange(len(sequences))
    
    # Embaralha aleatoriamente os índices
    np.random.shuffle(indices)

    # Se subsample for especificado, limita o número de índices ao valor de subsample
    if subsample is not None:
        indices = indices[:subsample]
        
    # Calcula o tamanho do conjunto de treino com base na proporção especificada
    train_size = int(len(indices) * train_ratio)
    
    # Calcula o tamanho do conjunto de validação com base na proporção especificada
    valid_size = int(len(indices) * valid_ratio)
    
    # Seleciona os primeiros índices como o conjunto de treino
    train_indices = indices[:train_size]
    
    # Seleciona os próximos índices como o conjunto de validação
    valid_indices = indices[train_size : train_size + valid_size]
    
    # Usa os índices restantes como o conjunto de teste
    test_indices = indices[train_size + valid_size :]
    
    # Cria o conjunto de treino com base nos índices de treino
    train_sequences = [sequences[i] for i in train_indices]
    
    # Cria o conjunto de validação com base nos índices de validação
    valid_sequences = [sequences[i] for i in valid_indices]
    
    # Cria o conjunto de teste com base nos índices de teste
    test_sequences = [sequences[i] for i in test_indices]

    # Retorna as sequências divididas nos conjuntos de treino, validação e teste
    return train_sequences, valid_sequences, test_sequences

In [34]:
# Configuração de hiperparâmetros
config = {
    # RÁPIDO: para testes rápidos e desenvolvimento
    "QUICK": { 
        "epochs": 50,
        "batch_size": 32,
        "subsample" : 250,  
        "lr": 1e-4,
        "d_model": 64,
        "feedforward_dim": 256,
        "num_layers": 3,
    },
    # COMPLETO: para treinamento completo
    "FULL": { 
        "epochs": 500,
        "batch_size": 32,
        "subsample" : None,
        "lr": 1e-4,
        "d_model": 512,
        "feedforward_dim": 2048,
        "num_layers": 6,
    },
}

In [35]:
# Parâmetros para organização dos dados
MAX_SEQ_LEN = 202   
MODE = "QUICK"  
TRAIN_RATIO = 0.8  
VALID_RATIO = 0.1 

In [36]:
# Faz a divisão dos dados
seq_treino, seq_valid, seq_teste = split_sequences(sequences, 
                                                   TRAIN_RATIO, 
                                                   VALID_RATIO, 
                                                   subsample = config[MODE]["subsample"])

In [37]:
seq_treino[0:3]

['MPQMMPINWIFFLFFFICIFLIFNIMNYFIYEKKMHLINKFNQKKKLTFNWKW',
 'MATRAWVAAAVALNPQLLPLRSCSPTKSVSPAQRSASMGLRLRSGRPCLGKFVCRRAKNAGYEDYKFPDPIPEFAEQETSKFREHMAWRLEQKKEDYFGDHVEEIVDVCTEIMGTFLENDYRGPGTLLVHPFLDMKGEIKERGLPGAPQAARAAIAWAEKNVDKDWKAWTGEY',
 'MMEEEECGLGKSCARSEPVAAAQPAGSPGETPAVAAESPELANYSSKCVFCRIAAHQDPGTELLHCENEDLVCFKDIKPAAPHHYLVVPKKHFENCKYLKKDQIELIENMVTVGKAILERNNFTDFENTRMGFHVSPFCSIAHLHLHVLAPADQLSFMSRLVYRVNSYWFITADYLIEKLRT']

### Criação dos Datasets e Dataloaders

In [38]:
# Cria os datasets
dataset_treino = ProteinDataset(seq_treino, token2idx)
dataset_valid = ProteinDataset(seq_valid, token2idx)
dataset_teste = ProteinDataset(seq_teste, token2idx)

In [39]:
print(*dataset_treino.data[0])
print(*dataset_teste.data[0])

<bos> M P Q M M P I N W I F F L F F F I C I F L I F N I M N Y F I Y E K K M H L I N K F N Q K K K L T F N W K W <eos>
<bos> M T D S D N P L I L V T G N K N K V L E V K A I L G P T A T L E V L D I N L P E I Q G S V E E I T R E K C R A A A E T I G G P V L V E D S A L E M R A L G G L P G A Y V K A F V E T I G N E G L N R I L S A F D D K S A E A V C T F G Y S Q G P G H E P L L F Q G R L Q G R I V P A R G V S S F G W E P I F E V E G E G V T L A E M E V G K K N G L S H R F K A L V K F R E W F L G A R R P V <eos>


In [40]:
# Cria o dataloader de treino
dl_treino = DataLoader(dataset_treino, 
                       batch_size = config[MODE]["batch_size"], 
                       shuffle = True, 
                       collate_fn = dataset_treino.padding_batch)

# Cria o dataloader de validação
dl_valid = DataLoader(dataset_valid, 
                      batch_size = config[MODE]["batch_size"], 
                      shuffle = False, 
                      collate_fn = dataset_valid.padding_batch)

# Cria o dataloader de teste
dl_teste = DataLoader(dataset_teste, 
                      batch_size = config[MODE]["batch_size"], 
                      shuffle = False, 
                      collate_fn = dataset_teste.padding_batch)

## Construção do Modelo Transformer

https://arxiv.org/pdf/1706.03762

### Módulo de Scaled Dot Product Attention

O Módulo de Scaled Dot Product Attention permite que o modelo se concentre em partes específicas da entrada ao processar uma sequência de dados.

Ele calcula a atenção entre palavras ou tokens usando produtos escalares (dot product) entre vetores de consulta (query), chave (key) e valor (value). 

Esses produtos são escalonados pela raiz da dimensão dos vetores para evitar valores extremos e, em seguida, passam por uma função softmax para gerar pesos de atenção. 

Esses pesos indicam a importância de cada token em relação aos outros, permitindo que o modelo capture relações e contextos, essenciais para tarefas como tradução e análise de linguagem.

In [41]:
class ScaledDotProductAttention(nn.Module):
    
    # Inicializa a classe com o parâmetro d_model
    def __init__(self, d_model):
        
        # Inicializa a superclasse nn.Module
        super().__init__()
        
        # Armazena o valor de d_model, que é a dimensão do modelo
        self.d_model = d_model
        
        # Calcula o fator de escala como o inverso da raiz quadrada de d_model
        self.scale = d_model ** (-0.5)

    # Define o método forward para o cálculo do passo de atenção
    def forward(self, query, key, value, mask = None):
        
        # Calcula a matriz de pontuações de atenção usando a operação de produto escalar escalado
        scores = self.scale * (query @ key.permute(0, 2, 1))

        # Aplica uma máscara (se fornecida), definindo posições mascaradas para um valor negativo infinito
        if mask is not None:
            scores[mask == 0.0] = -float("inf")

        # Calcula os pesos de atenção aplicando softmax ao longo da última dimensão
        attention_scores = F.softmax(scores, dim = -1)

        # Calcula o valor de saída multiplicando os pesos de atenção pelo vetor de valores
        output = attention_scores @ value

        # Retorna o resultado da operação de atenção
        return output

### Módulo de Single Head Attention

O Módulo de Single Head Attention realiza o cálculo da atenção em uma única "cabeça" ou unidade de atenção, o que significa que ele foca em um único padrão de relacionamento entre tokens por vez. 

Nesse processo, ele usa três vetores **(query, key e value)** para cada token da sequência, calculando os produtos escalares entre query e key para medir a similaridade e determinar o peso de atenção de cada token em relação aos outros. 

Esses pesos são então aplicados ao vetor value para gerar uma combinação ponderada que representa o contexto relevante para cada token. 

No entanto, por ser uma única "cabeça", ele **captura um único padrão de atenção**, limitando a capacidade de observar múltiplos aspectos de contexto na sequência. Para uma visão mais ampla, os Transformers usam Multi-Head Attention, que aplica o mesmo processo em várias "cabeças" paralelamente.

In [42]:
class SingleHeadAttention(nn.Module):
    
    # Inicializa a classe com um parâmetro d_model
    def __init__(self, d_model):
        
        # Inicializa a superclasse nn.Module
        super().__init__()

        # Define uma camada linear para projetar a consulta (query) na dimensão do modelo
        self.Wq = nn.Linear(d_model, d_model)
        
        # Define uma camada linear para projetar a chave (key) na dimensão do modelo
        self.Wk = nn.Linear(d_model, d_model)
        
        # Define uma camada linear para projetar o valor (value) na dimensão do modelo
        self.Wv = nn.Linear(d_model, d_model)

        # Instancia um objeto de atenção com produto escalar escalado
        self.attention = ScaledDotProductAttention(d_model)

        # Define uma camada linear para projetar a saída da atenção de volta na dimensão do modelo
        self.Wo = nn.Linear(d_model, d_model)

    # Define o método forward para o cálculo do passo de atenção com uma única cabeça
    def forward(self, query, key, value, mask = None):
        
        # Aplica a camada linear para obter a consulta transformada Q
        Q = self.Wq(query)
        
        # Aplica a camada linear para obter a chave transformada K
        K = self.Wk(key)
        
        # Aplica a camada linear para obter o valor transformado V
        V = self.Wv(value)

        # Calcula a saída da atenção usando as consultas, chaves, valores e, opcionalmente, uma máscara
        attention_output = self.attention(Q, K, V, mask = mask)

        # Aplica a camada linear final para projetar a saída da atenção
        output = self.Wo(attention_output)

        # Retorna a saída final da camada de atenção de uma única cabeça
        return output

### Módulo da Camada Transformer

O Módulo da Camada Transformer é uma unidade básica da arquitetura Transformer, combinando duas operações principais: 
**Multi-Head Attention** e uma rede **feedforward totalmente conectada**.

- O **Multi-Head Attention** permite que o modelo aprenda múltiplos padrões de relacionamento entre tokens simultaneamente, capturando diferentes aspectos do contexto. O resultado dessa atenção passa por uma normalização (normalização de camada) e um residual connection, que ajuda a manter o fluxo de informações ao evitar perda de características importantes.

- Após isso, uma rede **feedforward** é aplicada para processar e refinar a informação extraída, passando também por normalização e residual connection para estabilizar o aprendizado.

Essas duas operações juntas permitem que a camada Transformer identifique relações complexas entre os elementos de uma sequência, o que é essencial em tarefas de Processamento de Linguagem Natural e outras aplicações de séries temporais ou dados sequenciais.

In [43]:
class TransformerLayer(nn.Module):
    
    # Inicializa a classe com os parâmetros d_model, feedforward_dim e dropout
    def __init__(self, d_model, feedforward_dim, dropout = 0.1):
        
        # Inicializa a superclasse nn.Module
        super().__init__()

        # Armazena a dimensão do modelo
        self.d_model = d_model
        
        # Armazena a dimensão da camada de feedforward
        self.feedforward_dim = feedforward_dim

        # Instancia uma camada de atenção com uma única cabeça
        self.attention = SingleHeadAttention(d_model)

        # Define uma camada feedforward composta por duas camadas lineares e uma ReLU entre elas
        self.feed_forward = nn.Sequential(nn.Linear(d_model, feedforward_dim),
                                          nn.ReLU(),
                                          nn.Linear(feedforward_dim, d_model))

        # Define a primeira camada de normalização para o residual da atenção
        self.norm1 = nn.LayerNorm(d_model)
        
        # Define a segunda camada de normalização para o residual da camada feedforward
        self.norm2 = nn.LayerNorm(d_model)
        
        # Define uma camada de dropout para regularização
        self.dropout = nn.Dropout(dropout)

    # Define o método forward para o cálculo do passo do transformador
    def forward(self, x, mask = None):
        
        # Calcula a saída da atenção usando x como query, key e value
        attention_output = self.attention(x, x, x, mask = mask)

        # Aplica o dropout à saída da atenção, adiciona o residual de x e normaliza
        x = self.norm1(x + self.dropout(attention_output))

        # Passa a saída normalizada pela camada feedforward
        ff_output = self.feed_forward(x)

        # Aplica o dropout à saída feedforward, adiciona o residual de x e normaliza
        x = self.norm2(x + self.dropout(ff_output))

        # Retorna a saída final da camada de transformador
        return x

### Módulo Transformador

In [44]:
class Transformer(nn.Module):
    
    # Inicializa a classe com vários parâmetros, incluindo o tamanho do vocabulário, 
    # índices de padding, dimensões do modelo, etc.
    def __init__(self,
                 vocab_size,
                 pad_idx,
                 d_model,
                 feedforward_dim,
                 num_layers,
                 dropout,
                 device,
                 max_seq_len,
                 token2idx,
                 idx2token):

        # Inicializa a superclasse nn.Module
        super().__init__()
        
        # Armazena o índice de padding, o dispositivo, o comprimento máximo da sequência 
        # e os mapeamentos de tokens
        self.pad_idx = pad_idx
        self.device = device
        self.max_seq_len = max_seq_len
        self.token2idx = token2idx
        self.idx2token = idx2token

        # Define a camada de embedding para mapear tokens para vetores de dimensão d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx = pad_idx, device = device)

        # Define a camada de embedding para codificar posições até max_seq_len
        self.pos_embedding = nn.Embedding(max_seq_len, d_model, padding_idx = pad_idx, device = device)

        # Cria uma lista de camadas do transformador, cada uma composta por atenção e feedforward
        self.transformer_layers = nn.ModuleList([TransformerLayer(d_model, feedforward_dim, dropout)
                                                 for _ in range(num_layers)])

        # Define uma camada linear de saída para transformar as representações finais em logits 
        # de tamanho vocab_size
        self.output_layer = nn.Linear(d_model, vocab_size)

        # Move o modelo para o dispositivo especificado
        self.to(device)

    # Define o método forward para o passo de inferência
    def forward(self, x_ids):

        # Converte os índices dos tokens em embeddings de entrada
        embedded_input = self.embedding(x_ids)

        # Cria posições de sequência e as converte em embeddings de posição
        positions = torch.arange(x_ids.shape[1], device = self.device)
        embedded_pos = self.pos_embedding(positions)

        # Soma os embeddings de entrada e de posição
        x = embedded_input + embedded_pos
        
        # Gera a máscara de atenção para ignorar os tokens de padding
        mask = get_mask(x_ids, self.pad_idx).to(x.device)

        # Passa a entrada por todas as camadas do transformador
        for i, transformer_layer in enumerate(self.transformer_layers):
            x = transformer_layer(x, mask = mask)

        # Aplica a camada linear para obter os logits de saída
        logits = self.output_layer(x)

        # Retorna os logits, que representam as pontuações para cada token no vocabulário
        return logits

    # Define um método para gerar uma sequência de tokens, útil para inferência
    def generate_sequence(self, max_length = None):

        # Coloca o modelo em modo de avaliação
        self.eval()
        
        # Define o comprimento máximo da sequência a ser gerada
        max_length = max_length if max_length is not None else self.max_seq_len

        # Inicia a sequência com o token de início <bos>
        input_ids = torch.LongTensor([[self.token2idx["<bos>"]]]).to(self.device)

        # Desativa a gravação do gradiente para economizar memória durante a geração
        with torch.no_grad():
            
            for _ in range(self.max_seq_len - 1):

                # Calcula os logits para a sequência atual
                logits = self(input_ids)
                
                # Obtém os logits do último token na sequência atual
                final_logits = logits[:, -1, :]
                
                # Calcula as probabilidades para o próximo token usando softmax
                probs = F.softmax(final_logits, dim = -1)

                # Amostra o próximo token com base nas probabilidades
                next_token = torch.multinomial(probs.squeeze(), 1)

                # Anexa o próximo token à sequência de entrada
                input_ids = torch.cat([input_ids, next_token.unsqueeze(1)], dim = 1)

                # Interrompe a geração se o token de fim <eos> for gerado
                if next_token == self.token2idx["<eos>"]:
                    break

        # Converte os índices de tokens para uma lista e obtém os tokens correspondentes
        input_ids = input_ids.squeeze(0).detach().cpu().numpy()
        sequence = [self.idx2token[i] for i in input_ids]
        
        # Retorna a sequência de tokens gerada
        return sequence

In [45]:
# Hiperparâmetros
d_model = 4
feedforward_dim = 8
num_layers = 2
max_seq_len = 5

In [46]:
# Cria instância da classe
transformer = Transformer(vocab_size, 
                          token2idx['<pad>'], 
                          d_model, 
                          feedforward_dim, 
                          num_layers, 
                          0.1, 
                          device, 
                          max_seq_len, 
                          token2idx, 
                          idx2token)

In [47]:
# dados de exemplo
input_ids = torch.randint(3, 10, (2, 5), dtype = torch.long).to(device)

In [48]:
output = transformer(input_ids)

In [49]:
print(output.shape)

torch.Size([2, 5, 27])


### Módulo de Avaliação

In [50]:
def avalia_modelo(model, loader, criterion, device, pad_idx, vocab_size):

    # Inicializa variáveis para acumular a perda, previsões corretas e o total de tokens processados
    running_loss, running_correct_preds, running_total_tokens = 0.0, 0, 0
    
    # Coloca o modelo em modo de avaliação
    model.eval()
    
    # Desativa o cálculo do gradiente para economizar memória durante a avaliação
    with torch.no_grad():
        
        # Itera sobre os lotes de dados no loader
        for batch in loader:
            
            # Move os inputs e targets para o dispositivo (CPU ou GPU)
            input_ids = batch[0].to(device)
            target_ids = batch[1].to(device)

            # Faz uma previsão usando o modelo
            logits = model(input_ids)
            
            # Calcula a perda entre a previsão e os targets
            loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            # Acumula a perda ponderada pelo número de sequências no lote
            running_loss += loss.item() * input_ids.shape[0]
            
            # Cria uma máscara para ignorar os tokens de padding
            mask_pad = input_ids != pad_idx
            
            # Conta as previsões corretas ignorando os tokens de padding
            running_correct_preds += torch.sum(torch.argmax(logits, dim=-1)[mask_pad] == target_ids[mask_pad]).item()
            
            # Atualiza o total de tokens processados (excluindo padding)
            running_total_tokens += torch.sum(mask_pad).item()

    # Calcula a perda média para o conjunto de dados completo
    loss = running_loss / len(loader.dataset)
    
    # Calcula a acurácia para o conjunto de dados completo
    acc = running_correct_preds / running_total_tokens
    
    # Retorna a perda e a acurácia
    return loss, acc

### Módulo de Treinamento

In [51]:
def treina_modelo(epochs,
                  model,
                  train_loader,
                  valid_loader,
                  criterion,
                  optimizer,
                  device,
                  pad_idx,
                  vocab_size):
    
    # Armazenar o histórico de métricas durante o treinamento
    history = {"epoch": 0, "loss": [], "ppl": [], "acc": [], "val-loss": [], "val-ppl": [], "val-acc": []}
    
    # Itera por cada época de treinamento
    for epoch in range(1, epochs + 1):
        
        # Armazena o tempo de início da época
        start_epoch = time.time()

        # Inicializa variáveis para acumular perdas, número de sequências, previsões corretas e tokens totais
        running_loss, running_total_seq, running_correct_preds, running_total_tokens = (0.0, 0, 0, 0)
        
        # Coloca o modelo em modo de treino
        model.train()
        
        # Itera sobre os lotes de dados de treino
        for i, batch in enumerate(train_loader):
            
            # Zera os gradientes do otimizador
            optimizer.zero_grad()

            # Move os inputs e targets para o dispositivo (CPU ou GPU)
            input_ids = batch[0].to(device)
            target_ids = batch[1].to(device)

            # Faz uma previsão usando o modelo
            logits = model(input_ids)
            
            # Calcula a perda entre a previsão e os targets
            loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            # Acumula a perda ponderada pelo número de sequências
            running_loss += loss.item() * input_ids.shape[0]
            
            # Atualiza o total de sequências processadas
            running_total_seq += input_ids.shape[0]
            
            # Cria uma máscara para ignorar os tokens de padding
            mask_pad = input_ids != pad_idx
            
            # Conta as previsões corretas ignorando os tokens de padding
            running_correct_preds += torch.sum(torch.argmax(logits, dim = -1)[mask_pad] == target_ids[mask_pad]).item()
            
            # Atualiza o total de tokens processados (excluindo padding)
            running_total_tokens += torch.sum(mask_pad).item()

            # Realiza o backpropagation para calcular os gradientes
            loss.backward()
            
            # Atualiza os parâmetros do modelo usando o otimizador
            optimizer.step()

            # Limita a quantidade de lotes processados para cada época (caso i > 250)
            if i > 250: 
                break

        # Calcula a perda média de treino para a época
        train_loss = running_loss / running_total_seq
        
        # Calcula a acurácia de treino para a época
        train_acc = running_correct_preds / running_total_tokens

        # Avalia o modelo no conjunto de validação, calculando perda e acurácia
        val_loss, val_acc = avalia_modelo(model, valid_loader, criterion, device, pad_idx, vocab_size)

        # Calcula a perplexidade para treino e validação (exponenciação da perda média)
        train_ppl = math.exp(train_loss)
        val_ppl = math.exp(val_loss)
        
        # Atualiza o histórico de métricas para a época atual
        history["epoch"] += 1
        history["loss"].append(train_loss)
        history["ppl"].append(train_ppl)
        history["acc"].append(train_acc)
        history["val-loss"].append(val_loss)
        history["val-ppl"].append(val_ppl)
        history["val-acc"].append(val_acc)
        
        # Imprime o resumo das métricas para a época atual
        print(
            f"Epoch: {epoch}/{epochs} - loss={train_loss:.4f} - ppl={train_ppl:.4f} - acc={train_acc:.4f} - val-loss={val_loss:.4f} - val-ppl={val_ppl:.4f} - val-acc: {val_acc:.4f} ({time.time()-start_epoch:.2f}s/epoch)"
        )

    # Indica o fim do treinamento
    print("\nTreinamento Concluído.")
    
    # Retorna o histórico de métricas coletadas durante o treinamento
    return history

### Loop de Treinamento

In [52]:
# Cria o modelo
modelo = Transformer(vocab_size, 
                     pad_idx = token2idx['<pad>'],
                     d_model = config[MODE]["d_model"], 
                     feedforward_dim = config[MODE]["feedforward_dim"], 
                     num_layers = config[MODE]["num_layers"], 
                     dropout = 0.1, 
                     device = device,
                     max_seq_len = MAX_SEQ_LEN,
                     token2idx = token2idx,
                     idx2token = idx2token)

In [53]:
# Define o otimizador
optimizer = torch.optim.AdamW(modelo.parameters(), lr = config[MODE]["lr"])

In [54]:
# Define a função de erro
criterion = nn.CrossEntropyLoss(ignore_index = token2idx['<pad>']).to(device)  

In [ ]:
print(f"> Treinamento Iniciado em Modo: ({MODE})")
s = time.time()
history = treina_modelo(config[MODE]["epochs"], 
                          modelo, 
                          dl_treino, 
                          dl_valid, 
                          criterion, 
                          optimizer, 
                          device, 
                          token2idx['<pad>'], 
                          vocab_size
)
print(f"> Treinamento Finalizado em ({time.time() - s:.2f}s)")

### Avaliação e uso do modelo

In [ ]:
# plot
fig, axs = plt.subplots(1, 3,  figsize = (12, 4))
axs[0].set_title('Loss')
axs[0].plot(history['loss'], label = 'train')
axs[0].plot(history['val-loss'], label = 'valid')
axs[1].set_title('Perplexity')
axs[1].plot(history['ppl'], label = 'train')
axs[1].plot(history['val-ppl'], label = 'valid')
axs[2].set_title('Accuracy')
axs[2].plot(history['acc'], label = 'train')
axs[2].plot(history['val-acc'], label = 'valid')
for ax in axs.flat:
    ax.set(xlabel = 'Epoch')
    ax.grid()
    ax.legend();

In [ ]:
# Coloca o modelo em modo de avaliação
modelo.eval()

# Itera sobre os conjuntos de dados de treino e validação para avaliação
for name, dataset in [("Treino", dataset_treino), ("Validação", dataset_valid)]:
    
    # Escolhe um índice aleatório de uma sequência no conjunto de dados atual
    idx = np.random.choice(len(dataset))
    
    # Extrai a sequência de input_ids e target_ids com base no índice selecionado
    input_ids, target_ids = dataset[idx]
    
    # Desativa o cálculo do gradiente, pois é uma etapa de avaliação
    with torch.no_grad():
        
        # Faz uma previsão usando o modelo e converte o input_ids para um tensor de LongTensor 
        # no dispositivo apropriado
        logits = modelo(torch.LongTensor([input_ids]).to(device))
    
    # Converte os logits para previsões discretas, usando argmax para selecionar o índice de maior pontuação
    preds = torch.argmax(logits, dim = -1).squeeze(0).detach().cpu().numpy()
    
    # Imprime o nome do conjunto de dados e o índice da sequência avaliada
    print(f"\nSequência de Proteína em {name} com índice {idx}:\n")
    
    # Converte os target_ids de volta para tokens e os exibe
    print("  - Target:   ", " ".join([idx2token[i] for i in target_ids]))
    
    # Converte as previsões para tokens e os exibe
    print("\n  - Previsão:", " ".join([idx2token[i] for i in preds]))
    
    # Calcula a acurácia da previsão comparando com os target_ids
    print("\n  - Acurácia:", np.mean(preds == target_ids))

In [76]:
seq = modelo.generate_sequence()
print(f"Sequência Gerada de Comprimento {len(seq)}:")
print(*seq)

Sequência Gerada de Comprimento 182:
<bos> M A Q V V P A T P K L K N K V E V G L V P A V L R N F V N N H Q F G P A T F L K G E T Y V W E I V Q C S C Q G S G K K D F S V N V Q V I A A E S L Y D H L P L V H L Y K I Y S Y G T P K F Y V M S A G L I N I R R N G K K E R R N K R Q A Y H C P A G A T Y Y P Q P F K G K Q I K C P A E C C R T P E I R Y E C I I Y P A E V T I E D I C L K V V D T H C W Y I L C K E <eos>


In [77]:
# Avalia o modelo
test_loss, test_acc = avalia_modelo(modelo, dl_teste, criterion, device, token2idx['<pad>'], vocab_size)

A Perplexidade é uma métrica usada para avaliar modelos de linguagem, como redes neurais baseadas em Transformers, que predizem sequências de texto. Ela mede o quão bem o modelo é capaz de prever uma sequência de palavras e pode ser interpretada como uma medida de incerteza do modelo em relação às suas previsões.

A perplexidade é diretamente ligada à função de perda de entropia cruzada (Cross-Entropy Loss). De fato, a perplexidade é exponencial à entropia cruzada.

In [78]:
# Exponencial do erro
test_ppl = math.exp(test_loss)

In [79]:
print(f"Erro={test_loss:.4f} - Perplexidade={test_ppl:.4f} - Acurácia={test_acc:.4f}")

Erro=2.8867 - Perplexidade=17.9334 - Acurácia=0.1207


A perplexidade apresentou um valor elevado, indicando que o modelo ainda possui dificuldades na previsão. A acurácia também foi baixa. Vale ressaltar que o modelo foi treinado no modo *quick*, apenas para fins de teste inicial.